# 🌲 Random Forest & Bagging — Solutions Notebook

**Difficulty**: ⭐ Beginner  
**Time**: ~45 mins  
**Complete, verified reference implementation.**

---


## 🎯 Section 1: Overview

Bagging (Bootstrap Aggregation) trains trees in parallel on bootstrap samples. Random Forest adds feature sub-sampling at each split node to reduce tree correlation.

### Variance Reduction:
$$\text{Var}(\bar{X}) = \rho \sigma^2 + \frac{1-\rho}{M} \sigma^2$$
Random feature selection reduces $\rho$, lowering ensemble variance.


## 🔧 Section 2: Implementation from Scratch


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete! ✅')

In [ ]:
from sklearn.tree import DecisionTreeClassifier

class RandomForestFromScratch:
    def __init__(self, n_estimators=10, max_features='sqrt', max_depth=3):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.max_depth = max_depth
        self.trees = []
        self.feature_indices = []
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        k_features = int(np.sqrt(n_features)) if self.max_features == 'sqrt' else n_features
        
        self.trees = []
        self.feature_indices = []
        
        for _ in range(self.n_estimators):
            # Bootstrap sample
            boot_idx = np.random.choice(n_samples, n_samples, replace=True)
            X_boot, y_boot = X[boot_idx], y[boot_idx]
            
            # Feature subset
            feat_idx = np.random.choice(n_features, k_features, replace=False)
            
            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=42)
            tree.fit(X_boot[:, feat_idx], y_boot)
            
            self.trees.append(tree)
            self.feature_indices.append(feat_idx)
        return self

    def predict(self, X):
        tree_preds = np.array([
            tree.predict(X[:, feat_idx]) 
            for tree, feat_idx in zip(self.trees, self.feature_indices)
        ])
        # Majority voting across trees
        return np.array([np.bincount(tree_preds[:, i]).argmax() for i in range(X.shape[0])])


In [ ]:
X = np.random.rand(100, 4)
y = np.random.randint(0, 2, 100)
rf = RandomForestFromScratch(n_estimators=5)
rf.fit(X, y)
preds = rf.predict(X)
print('Predictions shape:', preds.shape)


## 📦 Section 3: Library Implementation


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, max_features='sqrt', oob_score=True, random_state=42)
rf.fit(X_train, y_train)
print(f"OOB Accuracy: {rf.oob_score_:.4f}")


## ❓ Section 4: Interview Questions


### Q1: How does Random Forest prevent overfitting?
**Answer**: By combining Bootstrap Aggregation (averaging predictions of independent trees) and random feature sub-sampling, which lowers pairwise correlation between trees.


### Q2: What is Out-Of-Bag (OOB) error?
**Answer**: About 36.8% of samples are not selected in each bootstrap sample. OOB error evaluates performance on these left-out instances without needing a separate cross-validation split.
